<a href="https://colab.research.google.com/github/diomani-ouattara/Portfolio-Data-scientist/blob/main/Happinesscheck2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ACME Customer Happiness Survey 2020 — Predictive Analysis

In [63]:
# Import Librairies
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_curve, roc_auc_score
)
from sklearn.feature_selection import mutual_info_classif

import warnings
warnings.filterwarnings('ignore')

SEED = 42

In [4]:
# ─── 1. LOAD DATA ───────────────────────────────────────────────

from google.colab import files
print("Please upload  ACME-HappinessSurvey2020.csv")
uploaded = files.upload()
import io
df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]))


Please upload  ACME-HappinessSurvey2020.csv


Saving ACME-HappinessSurvey2020.csv to ACME-HappinessSurvey2020 (1).csv


In [5]:
# Features Labels
feature_labels = {
    'X1': 'Order delivered on time',
    'X2': 'Contents were as expected',
    'X3': 'Ordered everything I wanted',
    'X4': 'Paid a good price',
    'X5': 'Satisfied with courier',
    'X6': 'App makes ordering easy',
}

X = df.drop('Y', axis=1)
y = df['Y']


**CHECKING THE QUALITY OF THE DATA (FORM,NULL VALUES, ETC)**

In [6]:
# Checking the number of rows and columns in the training data
df.shape

(126, 7)

In [7]:
# let's view the first 5 rows of the data
df.head()

,Y,X1,X2,X3,X4,X5,X6
0,0,3,3,3,4,2,4
1,0,3,2,3,5,4,3
2,1,5,3,3,3,3,5
3,0,5,4,3,3,3,5
4,0,5,4,3,3,3,5


In [8]:
# let's view the last 5 rows of the data
df.tail()

,Y,X1,X2,X3,X4,X5,X6
121,1,5,2,3,4,4,3
122,1,5,2,3,4,2,5
123,1,5,3,3,4,4,5
124,0,4,3,3,4,4,5
125,0,5,3,2,5,5,5


In [9]:
# let's check for missing values in the data
round(df.isnull().sum() / df.isnull().count() * 100, 2)

,0
Y,0.0
X1,0.0
X2,0.0
X3,0.0
X4,0.0
X5,0.0
X6,0.0


In [10]:
# let's check the data types of the columns in the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Y       126 non-null    int64
 1   X1      126 non-null    int64
 2   X2      126 non-null    int64
 3   X3      126 non-null    int64
 4   X4      126 non-null    int64
 5   X5      126 non-null    int64
 6   X6      126 non-null    int64
dtypes: int64(7)
memory usage: 7.0 KB


In [38]:
# let's view the statistical summary of the numerical columns in the data
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Y,126.0,0.547619,0.499714,0.0,0.0,1.0,1.0,1.0
X1,126.0,4.333333,0.800000,1.0,4.0,5.0,5.0,5.0
X2,126.0,2.531746,1.114892,1.0,2.0,3.0,3.0,5.0
X3,126.0,3.309524,1.023440,1.0,3.0,3.0,4.0,5.0
X4,126.0,3.746032,0.875776,1.0,3.0,4.0,4.0,5.0
X5,126.0,3.650794,1.147641,1.0,3.0,4.0,4.0,5.0
X6,126.0,4.253968,0.809311,1.0,4.0,4.0,5.0,5.0


2. **EXPLORATORY DATA ANALYSIS (EDA)**

In [64]:


# ── 2a. Class Distribution ──────────────────────────────────────
class_counts = y.value_counts().sort_index()
fig_cls = go.Figure(go.Bar(
    x=['Unhappy (0)', 'Happy (1)'],
    y=class_counts.values,
    marker_color=['#e05c5c', '#4caf7d'],
    text=[f'{v}<br>({v/len(y)*100:.1f}%)' for v in class_counts.values],
    textposition='outside',
    cliponaxis=False,
    width=0.4,
))
fig_cls.update_layout(
    title='Class Distribution — Happy vs Unhappy Customers',
    yaxis_title='Number of Customers',
    yaxis_range=[0, class_counts.max() * 1.2],
    width=480, height=380,
    margin=dict(t=60, b=40),
)
fig_cls.show()


In [53]:
# ── 2b. Response Distribution per Feature (1–5 scale) ───────────
features = list(feature_labels.keys())
fig_resp = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'{f}: {feature_labels[f]}' for f in features],
    vertical_spacing=0.18,
    horizontal_spacing=0.08,
)
score_colors = ['#d73027', '#fc8d59', '#fee090', '#91cf60', '#1a9850']

for idx, feat in enumerate(features):
    r, c = divmod(idx, 3)
    counts = df[feat].value_counts().sort_index()
    for score in range(1, 6):
        cnt = counts.get(score, 0)
        fig_resp.add_trace(
            go.Bar(
                x=[score], y=[cnt],
                marker_color=score_colors[score - 1],
                text=[cnt], textposition='outside',
                cliponaxis=False,
                name=f'Score {score}',
                showlegend=(idx == 0),
                legendgroup=f'score{score}',
            ),
            row=r + 1, col=c + 1,
        )
    fig_resp.update_xaxes(tickvals=list(range(1, 6)),
                          title_text='Rating', row=r + 1, col=c + 1)
    fig_resp.update_yaxes(title_text='Count' if c == 0 else '',
                          row=r + 1, col=c + 1)

fig_resp.update_layout(
    title='Response Distribution per Survey Question (Rating 1–5)',
    barmode='group',
    height=580,
    legend_title='Score',
    margin=dict(t=80, b=40),
)
fig_resp.show()


In [40]:
# ── 2c. Mean Score by Happiness Class ───────────────────────────
means = df.groupby('Y')[features].mean().T
means.columns = ['Unhappy', 'Happy']

fig_mean = go.Figure()
for cls, color in [('Unhappy', '#e05c5c'), ('Happy', '#4caf7d')]:
    fig_mean.add_trace(go.Bar(
        name=cls,
        x=[f'{f}: {feature_labels[f]}' for f in means.index],
        y=means[cls].round(2),
        marker_color=color,
        text=means[cls].round(2),
        textposition='outside',
        cliponaxis=False,
    ))
fig_mean.update_layout(
    title='Average Feature Score — Happy vs Unhappy Customers',
    barmode='group',
    yaxis=dict(title='Mean Rating (1–5)', range=[0, 6]),
    xaxis_tickangle=-20,
    height=430,
    legend_title='Class',
    margin=dict(t=60, b=120),
)
fig_mean.show()


In [41]:
# ── 2d. Box Plots — Score Distribution by Class ─────────────────
fig_box = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'{f}: {feature_labels[f]}' for f in features],
    vertical_spacing=0.18,
    horizontal_spacing=0.08,
)
box_colors = {0: '#e05c5c', 1: '#4caf7d'}
box_labels  = {0: 'Unhappy', 1: 'Happy'}

for idx, feat in enumerate(features):
    r, c = divmod(idx, 3)
    for cls in [0, 1]:
        fig_box.add_trace(
            go.Box(
                y=df[df['Y'] == cls][feat],
                name=box_labels[cls],
                marker_color=box_colors[cls],
                boxmean=True,
                showlegend=(idx == 0),
                legendgroup=box_labels[cls],
            ),
            row=r + 1, col=c + 1,
        )
    fig_box.update_yaxes(range=[0, 6], tickvals=list(range(1, 6)),
                         row=r + 1, col=c + 1)

fig_box.update_layout(
    title='Score Distribution by Class — Box Plots (dashed line = mean)',
    height=580,
    legend_title='Class',
    margin=dict(t=80, b=40),
    boxmode='group',
)
fig_box.show()


In [42]:
# ── 2e. Correlation Heatmap ──────────────────────────────────────
corr_matrix = df.rename(columns={**feature_labels, 'Y': 'Happy (Y)'}).corr().round(2)
fig_heat = go.Figure(go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    text=corr_matrix.values,
    texttemplate='%{text}',
    textfont_size=11,
    colorscale='RdYlGn',
    zmid=0,
    zmin=-1, zmax=1,
    hovertemplate='%{y} × %{x}: %{z}<extra></extra>',
))
fig_heat.update_layout(
    title='Correlation Heatmap — All Features + Target',
    width=620, height=520,
    margin=dict(t=60, b=100, l=160),
    xaxis_tickangle=-30,
)
fig_heat.show()


**3.Predictive Modeling**

In [62]:
# ─── 3. FEATURE SELECTION ───────────────────────────────────────
# Three independent methods; lowest average rank → most important

# Method 1: Pearson correlation with target
corr  = X.apply(lambda col: abs(col.corr(y)))

# Method 2: Mutual Information
mi    = mutual_info_classif(X, y, random_state=SEED)

# Method 3: Random Forest feature importance (trained on full data)
rf_selector = RandomForestClassifier(n_estimators=300, random_state=SEED)
rf_selector.fit(X, y)
rf_imp = rf_selector.feature_importances_

ranking = pd.DataFrame({
    'Feature'   : X.columns,
    'Label'     : [feature_labels[f] for f in X.columns],
    'Correlation': corr.values,
    'Mut_Info'  : mi,
    'RF_Imp'    : rf_imp,
})

# Rank within each method (1 = most important)
for col in ['Correlation', 'Mut_Info', 'RF_Imp']:
    ranking[f'{col}_rank'] = ranking[col].rank(ascending=False)

ranking['Avg_Rank'] = ranking[['Correlation_rank',
                                'Mut_Info_rank',
                                'RF_Imp_rank']].mean(axis=1)
ranking = ranking.sort_values('Avg_Rank').reset_index(drop=True)

print("\n" + "=" * 60)
print("FEATURE RANKING  (lower avg rank = more predictive)")
print("=" * 60)
print(ranking[['Feature', 'Label', 'Avg_Rank',
               'Correlation', 'Mut_Info', 'RF_Imp']
             ].to_string(index=False, float_format='{:.4f}'.format))

TOP_3    = ranking['Feature'].head(3).tolist()
BOTTOM_3 = ranking['Feature'].tail(3).tolist()

print(f"\n  KEEP   (top 3)    : {TOP_3}")
print(f"  REMOVE (bottom 3) : {BOTTOM_3}")


# ─── 3. FEATURE IMPORTANCE CHART  ─────────
methods = [
    ('Correlation', '|Pearson r|'),
    ('Mut_Info',    'Mutual Information'),
    ('RF_Imp',      'Random Forest Importance'),
]

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=[m[1] for m in methods],
                    horizontal_spacing=0.12)

for col_idx, (col, xlabel) in enumerate(methods, start=1):
    sorted_df = ranking.sort_values(col)
    colors = ['#e05c5c' if f in BOTTOM_3 else '#4caf7d'
              for f in sorted_df['Feature']]
    fig.add_trace(
        go.Bar(
            x=sorted_df[col],
            y=sorted_df['Feature'],
            orientation='h',
            marker_color=colors,
            text=sorted_df[col].round(3),
            textposition='outside',
            cliponaxis=False,
            hovertemplate='%{y}: %{x:.4f}<extra></extra>',
            showlegend=False,
        ),
        row=1, col=col_idx
    )
    # Pad the x-axis so outside labels are never clipped
    max_val = sorted_df[col].max()
    fig.update_xaxes(range=[0, max_val * 1.35], row=1, col=col_idx)

# Dummy traces for legend
for label, color in [('Keep (top 3)', '#4caf7d'), ('Remove (bottom 3)', '#e05c5c')]:
    fig.add_trace(go.Bar(x=[None], y=[None], orientation='h',
                         marker_color=color, name=label,
                         showlegend=True))

fig.update_layout(
    title_text='Feature Selection — Three Methods',
    title_font_size=15,
    height=380,
    legend=dict(orientation='h', yanchor='bottom', y=-0.25, xanchor='center', x=0.5),
    margin=dict(t=80, b=80),
)
fig.show()




FEATURE RANKING  (lower avg rank = more predictive)
Feature                       Label  Avg_Rank  Correlation  Mut_Info  RF_Imp
     X1     Order delivered on time    2.0000       0.2802    0.0496  0.1674
     X5      Satisfied with courier    2.3333       0.2245    0.0399  0.1804
     X3 Ordered everything I wanted    3.3333       0.1508    0.0000  0.1863
     X2   Contents were as expected    3.6667       0.0243    0.0045  0.1819
     X6     App makes ordering easy    4.6667       0.1677    0.0000  0.1320
     X4           Paid a good price    5.0000       0.0644    0.0000  0.1521

  KEEP   (top 3)    : ['X1', 'X5', 'X3']
  REMOVE (bottom 3) : ['X2', 'X6', 'X4']


In [13]:
# Rank within each method (1 = most important)
for col in ['Correlation', 'Mut_Info', 'RF_Imp']:
    ranking[f'{col}_rank'] = ranking[col].rank(ascending=False)

ranking['Avg_Rank'] = ranking[['Correlation_rank',
                                'Mut_Info_rank',
                                'RF_Imp_rank']].mean(axis=1)
ranking = ranking.sort_values('Avg_Rank').reset_index(drop=True)

print("\n" + "=" * 60)
print("FEATURE RANKING  (lower avg rank = more predictive)")
print("=" * 60)
print(ranking[['Feature', 'Label', 'Avg_Rank',
               'Correlation', 'Mut_Info', 'RF_Imp']
             ].to_string(index=False, float_format='{:.4f}'.format))

TOP_3    = ranking['Feature'].head(3).tolist()
BOTTOM_3 = ranking['Feature'].tail(3).tolist()

print(f"\n  KEEP   (top 3)    : {TOP_3}")
print(f"  REMOVE (bottom 3) : {BOTTOM_3}")




FEATURE RANKING  (lower avg rank = more predictive)
Feature                       Label  Avg_Rank  Correlation  Mut_Info  RF_Imp
     X1     Order delivered on time    2.0000       0.2802    0.0496  0.1674
     X5      Satisfied with courier    2.3333       0.2245    0.0399  0.1804
     X3 Ordered everything I wanted    3.3333       0.1508    0.0000  0.1863
     X2   Contents were as expected    3.6667       0.0243    0.0045  0.1819
     X6     App makes ordering easy    4.6667       0.1677    0.0000  0.1320
     X4           Paid a good price    5.0000       0.0644    0.0000  0.1521

  KEEP   (top 3)    : ['X1', 'X5', 'X3']
  REMOVE (bottom 3) : ['X2', 'X6', 'X4']


In [65]:
# ─── 4. TRAIN / TEST SPLIT  (80 / 20) ───────────────────────────
X_full    = X[ranking['Feature'].tolist()]   # all 6 features
X_reduced = X[TOP_3]                          # top 3 only

X_tr_f, X_te_f, y_tr, y_te = train_test_split(
    X_full, y, test_size=0.2, random_state=SEED, stratify=y)

X_tr_r = X_tr_f[TOP_3]
X_te_r = X_te_f[TOP_3]

# Scale for distance/regression models
sc_f, sc_r = StandardScaler(), StandardScaler()
X_tr_f_s = sc_f.fit_transform(X_tr_f);  X_te_f_s = sc_f.transform(X_te_f)
X_tr_r_s = sc_r.fit_transform(X_tr_r);  X_te_r_s = sc_r.transform(X_te_r)

print(f"\nTrain size : {len(y_tr)}  |  Test size : {len(y_te)}")



Train size : 100  |  Test size : 26


In [66]:
# ─── 5. MODEL COMPARISON ────────────────────────────────────────
# Note: No cross-validation on the 3-feature model (small dataset)

MODELS = {
    'Logistic Regression': LogisticRegression(random_state=SEED, max_iter=1000, C=1.0),
    'Decision Tree'      : DecisionTreeClassifier(random_state=SEED, max_depth=4),
    'Random Forest'      : RandomForestClassifier(random_state=SEED, n_estimators=300),
    'KNN'                : KNeighborsClassifier(n_neighbors=5),
}

NEEDS_SCALING = {'Logistic Regression', 'KNN'}

rows = []
for name, model in MODELS.items():
    scaled = name in NEEDS_SCALING

    # — All 6 features
    model.fit(X_tr_f_s if scaled else X_tr_f, y_tr)
    p6 = model.predict(X_te_f_s if scaled else X_te_f)

    # — Top 3 features (no CV — per project instructions)
    model.fit(X_tr_r_s if scaled else X_tr_r, y_tr)
    p3 = model.predict(X_te_r_s if scaled else X_te_r)

    rows.append({
        'Model'        : name,
        'Acc (6 feat)' : round(accuracy_score(y_te, p6), 4),
        'F1  (6 feat)' : round(f1_score(y_te, p6), 4),
        'Acc (3 feat)' : round(accuracy_score(y_te, p3), 4),
        'F1  (3 feat)' : round(f1_score(y_te, p3), 4),
    })

results_df = pd.DataFrame(rows).sort_values('Acc (3 feat)', ascending=False)

print("\n" + "=" * 62)
print("MODEL COMPARISON")
print("=" * 62)
print(results_df.to_string(index=False))
print(f"\nTarget accuracy : ≥ 73%")

# ── Transition chart: 6 features → 3 features ───────────────────
TARGET = 0.73
model_names = results_df['Model'].tolist()
acc_6 = results_df['Acc (6 feat)'].tolist()
acc_3 = results_df['Acc (3 feat)'].tolist()

fig_trans = go.Figure()

# 6-feature bars
fig_trans.add_trace(go.Bar(
    name='6 Features (all)',
    x=model_names,
    y=acc_6,
    marker_color='#aec6e8',
    text=[f'{v*100:.1f}%' for v in acc_6],
    textposition='outside',
    cliponaxis=False,
))

# 3-feature bars — colour green if ≥73%, red if below
colors_3 = ['#4caf7d' if v >= TARGET else '#e05c5c' for v in acc_3]
fig_trans.add_trace(go.Bar(
    name='3 Features (reduced)',
    x=model_names,
    y=acc_3,
    marker_color=colors_3,
    text=[f'{v*100:.1f}%' for v in acc_3],
    textposition='outside',
    cliponaxis=False,
))

# 73% target line
fig_trans.add_hline(
    y=TARGET,
    line_dash='dash',
    line_color='crimson',
    line_width=2,
    annotation_text='73% target',
    annotation_position='top right',
    annotation_font_color='crimson',
)

fig_trans.update_layout(
    title='Accuracy: 6 Features vs 3 Features — Can We Hit 73%?',
    barmode='group',
    yaxis=dict(
        title='Accuracy',
        tickformat='.0%',
        range=[0, 1.05],
    ),
    xaxis_title='Model',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    height=440,
    margin=dict(t=80, b=40),
)
fig_trans.show()



MODEL COMPARISON
              Model  Acc (6 feat)  F1  (6 feat)  Acc (3 feat)  F1  (3 feat)
      Decision Tree        0.8077        0.8485        0.7692        0.7692
                KNN        0.6154        0.7059        0.7692        0.7692
      Random Forest        0.6538        0.6667        0.7308        0.7586
Logistic Regression        0.6154        0.7059        0.6154        0.7059

Target accuracy : ≥ 73%


In [67]:
# ─── 6. BEST MODEL — FULL EVALUATION ────────────────────────────
best_name  = results_df.iloc[0]['Model']
best_model = MODELS[best_name]
scaled     = best_name in NEEDS_SCALING

best_model.fit(X_tr_r_s if scaled else X_tr_r, y_tr)
y_pred = best_model.predict(X_te_r_s if scaled else X_te_r)

acc  = accuracy_score(y_te, y_pred)
f1   = f1_score(y_te, y_pred)
prec = precision_score(y_te, y_pred)
rec  = recall_score(y_te, y_pred)

print("\n" + "=" * 55)
print(f"BEST MODEL : {best_name}")
print(f"FEATURES   : {TOP_3}  →  {[feature_labels[f] for f in TOP_3]}")
print("=" * 55)
print(f"  Accuracy  : {acc*100:.1f}%  {'✓  TARGET MET' if acc >= 0.73 else '✗  below target'}")
print(f"  F1 Score  : {f1:.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print("\nFull Classification Report:")
print(classification_report(y_te, y_pred,
                            target_names=['Unhappy (0)', 'Happy (1)']))



BEST MODEL : Decision Tree
FEATURES   : ['X1', 'X5', 'X3']  →  ['Order delivered on time', 'Satisfied with courier', 'Ordered everything I wanted']
  Accuracy  : 76.9%  ✓  TARGET MET
  F1 Score  : 0.7692
  Precision : 0.8333
  Recall    : 0.7143

Full Classification Report:
              precision    recall  f1-score   support

 Unhappy (0)       0.71      0.83      0.77        12
   Happy (1)       0.83      0.71      0.77        14

    accuracy                           0.77        26
   macro avg       0.77      0.77      0.77        26
weighted avg       0.78      0.77      0.77        26



In [69]:
# Confusion Matrix (interactive — Plotly)
cm = confusion_matrix(y_te, y_pred)
labels = ['Unhappy', 'Happy']
cm_text = [[f'<b>{v}</b>' for v in row] for row in cm]

fig_cm = go.Figure(go.Heatmap(
    z=cm,
    x=labels,
    y=labels,
    text=cm_text,
    texttemplate='%{text}',
    textfont_size=20,
    colorscale='Blues',
    showscale=False,
    hovertemplate='Actual: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>',
))
fig_cm.update_layout(
    title=f'Confusion Matrix — {best_name}<br>Features: {TOP_3}',
    xaxis_title='Predicted',
    yaxis_title='Actual',
    width=400, height=380,
    margin=dict(t=80),
)
fig_cm.show()


In [70]:
# ROC Curve — all models on the 3 selected features (Plotly)
fig_roc = go.Figure()

# Diagonal reference line (random classifier)
fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    line=dict(color='grey', dash='dash', width=1.5),
    name='Random classifier (AUC = 0.50)',
    hoverinfo='skip',
))

roc_colors = {
    'Logistic Regression': '#1f77b4',
    'Decision Tree'      : '#ff7f0e',
    'Random Forest'      : '#2ca02c',
    'KNN'                : '#9467bd',
}

for name, model in MODELS.items():
    scaled = name in NEEDS_SCALING
    Xtr = X_tr_r_s if scaled else X_tr_r
    Xte = X_te_r_s if scaled else X_te_r

    model.fit(Xtr, y_tr)
    y_prob = model.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, y_prob)
    auc = roc_auc_score(y_te, y_prob)

    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f'{name} (AUC = {auc:.3f})',
        line=dict(color=roc_colors[name], width=2.5),
        hovertemplate='FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra>' + name + '</extra>',
    ))

fig_roc.update_layout(
    title=f'ROC Curves — All Models on 3 Selected Features {TOP_3}',
    xaxis=dict(title='False Positive Rate', range=[-0.02, 1.02]),
    yaxis=dict(title='True Positive Rate', range=[-0.02, 1.05]),
    legend=dict(yanchor='bottom', y=0.05, xanchor='right', x=0.98),
    width=600, height=500,
    margin=dict(t=70),
)
fig_roc.show()



In [71]:
# ─── 7. BUSINESS SUMMARY ────────────────────────────────────────
print("\n" + "=" * 60)
print("BUSINESS SUMMARY")
print("=" * 60)
print(f"\n  Survey questions to KEEP for next survey:")
for i, f in enumerate(TOP_3, 1):
    print(f"    {i}. {f} — {feature_labels[f]}")

print(f"\n  Survey questions that can be REMOVED:")
for i, f in enumerate(BOTTOM_3, 1):
    print(f"    {i}. {f} — {feature_labels[f]}")

print(f"""
  Model   : {best_name}
  Accuracy: {acc*100:.1f}%  (target ≥ 73%)

  Interpretation:
  ─ Customers who rate the retained questions highly are
    significantly more likely to be classified as happy.
  ─ The 3 removed questions add minimal predictive signal
    and can be dropped from the next survey without losing
    meaningful classification power.
""")


BUSINESS SUMMARY

  Survey questions to KEEP for next survey:
    1. X1 — Order delivered on time
    2. X5 — Satisfied with courier
    3. X3 — Ordered everything I wanted

  Survey questions that can be REMOVED:
    1. X2 — Contents were as expected
    2. X6 — App makes ordering easy
    3. X4 — Paid a good price

  Model   : Decision Tree
  Accuracy: 76.9%  (target ≥ 73%)

  Interpretation:
  ─ Customers who rate the retained questions highly are
    significantly more likely to be classified as happy.
  ─ The 3 removed questions add minimal predictive signal
    and can be dropped from the next survey without losing
    meaningful classification power.



Based on the data, the feature importance chart you shared, and the business context, here are my recommendations:

**Key Findings**
**X1 (on-time delivery)** is the single strongest predictor of happiness — it ranks #1 in both Pearson correlation and Mutual Information. This is your most controllable lever.

**X5 (courier satisfaction)**  is a close second — consistent across all three methods. The human element of delivery has a direct, measurable impact on happiness.

**X4 (price) and X2 (order contents)** are weak predictors — customers are not differentiating happy vs. unhappy based on price or whether the contents matched. This has two implications: you likely have pricing power, and product catalog accuracy, while important operationally, doesn't drive emotional satisfaction as much as you'd expect.

**Business Recommendations**
1. Prioritize on-time delivery above everything else
It's your #1 happiness driver. Any investment in routing optimization, delivery windows, or real-time tracking will have the highest ROI on customer happiness scores.

2. Invest in the courier experience
X5 is strong and consistent. Courier training, performance incentives, ratings systems, and courier-facing tools are worth the investment — they directly move the happiness needle.

3. Don't compete on price to drive happiness
X4 is the weakest predictor. Customers who are happy are not significantly more likely to say they got a good price. Loyalty comes from reliability and service quality, not discounts.

4. Shorten your next survey to 3 questions
The 3 removed features add noise without predictive value. A shorter survey will get higher response rates and the model loses minimal accuracy — a direct operational win.

5. Build an early-warning system
With X1 and X5 as your core signals, you can flag at-risk customers in near real-time after each delivery without waiting for survey responses — use delivery timestamps and courier ratings as proxies.

6. Segment your unhappy customers
The model predicts who is unhappy, but not why at an individual level. Cross-referencing predicted-unhappy customers with their X1/X5 scores will tell you whether the root cause is a logistics problem (routes, capacity) or a people problem (specific couriers, zones).